# Refined Benchmark

Two modes controlled by `LOAD_BASELINE`:
- `LOAD_BASELINE = True` — loads generated code + metrics from an existing baseline output dir, then runs analysis → refinement → evaluation. No initial LLM call.
- `LOAD_BASELINE = False` — runs the full pipeline: initial LLM call → evaluate → analyze → refine → evaluate.

In [1]:
import importlib
import evaluate as evaluate_module
import analysis as analysis_module
from evaluate import evaluate, print_metrics, print_evaluation_summary
from benchmarks import load_benchmarks
from llm import call_llm, build_prompt
from analysis import analyze
import output as output_module
from output import load_baseline_results

## Configuration

In [2]:
from pathlib import Path
BENCHMARK_DIR = Path("./benchmark")
BENCHMARK_TYPES = ["socbenchd_1"]  # add more benchmark types here
BENCHMARK_LIMIT = None  # set to None for all sectors
QUERY_LIMIT = 50        # set to None for all queries (ignored when LOAD_BASELINE=True)
MAX_WORKERS = 10
MODEL = "deepseek-ai/DeepSeek-V4-Pro"

LOAD_BASELINE = True   # True = load saved baseline from disk; False = run initial LLM call fresh
BASELINE_DIR = "output/2026-06-16_15-57-18_baseline"  # only used when LOAD_BASELINE=True

benchmark_sets, total_available, total_queries_available = load_benchmarks(BENCHMARK_DIR, BENCHMARK_TYPES, BENCHMARK_LIMIT, None)
total_queries = min(QUERY_LIMIT, total_queries_available) if QUERY_LIMIT is not None else total_queries_available
print(f"Loaded {len(benchmark_sets)} benchmark sets (total available: {total_available})")
if not LOAD_BASELINE:
    print(f"Total queries: {total_queries} (of {total_queries_available} available)")
print(f"Mode: {'LOAD BASELINE from ' + BASELINE_DIR if LOAD_BASELINE else 'FULL (run initial + refine)'} | Workers: {MAX_WORKERS} | Model: {MODEL}")

Loaded 11 benchmark sets (total available: 11)
Mode: LOAD BASELINE from output/2026-06-16_15-57-18_baseline | Workers: 10 | Model: deepseek-ai/DeepSeek-V4-Pro


## Prompt Template

Used when `LOAD_BASELINE = False`. When loading a baseline the prompt is read from `prompt.txt` in the baseline output directory.

In [3]:
PROMPT_TEMPLATE = '''You're doing a Service Composition.
You are given a set of REST API specifications and a task description.
Your job is to write Python code using the appropriate client library that fulfills the task by calling the necessary endpoints in the correct order. Import requests and create a function called compose.

Rules:
- Use the requests library.
- Only use endpoints defined in the provided specifications.
- Return ONLY raw Python code. No markdown, no code fences, no comments, no notes, no explanations — nothing but the code itself.
- Import the requests library and create a function called compose, where all the requests shall be called. Do NOT call that function.

source
{services_block}

## Task

{query}

## Python Code
'''


## Initial LLM Call  (or Load Baseline)

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed

if LOAD_BASELINE:
    importlib.reload(output_module)
    sector_results = load_baseline_results(BASELINE_DIR)
    sector_results.sort(key=lambda r: (r['sector_name'], r['query_index']))
    print(f"Loaded {len(sector_results)} results from {BASELINE_DIR}")
else:
    tasks = []
    for benchmark in benchmark_sets:
        if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
            break
        for query_index, query in enumerate(benchmark['queries'], start=1):
            if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
                break
            tasks.append((benchmark, query_index, query))

    def _call_initial(args):
        benchmark, query_index, query = args
        prompt = build_prompt(benchmark['services'], query['query'], PROMPT_TEMPLATE)
        generated = call_llm(prompt, MODEL, '')
        generated += '\n\ncompose()'
        return {
            'query_index': query_index,
            'sector_name': benchmark['name'],
            'query': query,
            'prompt': prompt,
            'generated': generated,
            'service_files': benchmark.get('service_files', []),
            'model': MODEL
        }

    sector_results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_call_initial, t): t for t in tasks}
        for future in as_completed(futures):
            result = future.result()
            sector_results.append(result)
            print(f"[{len(sector_results)}/{total_queries}] [{result['sector_name']}] Query {result['query_index']} done")

    sector_results.sort(key=lambda r: (r['sector_name'], r['query_index']))

Loaded 50 results from output/2026-06-16_15-57-18_baseline


## Evaluate Initial

Skipped when loading a baseline — metrics are already stored in the loaded results.

In [5]:
if not LOAD_BASELINE:
    for result in sector_results:
        initial_metrics = evaluate(result['generated'], result['query'].get('endpoints', []))
        result['initial_metrics'] = initial_metrics
        print_metrics(initial_metrics, f"Initial evaluation - Query {result['query_index']}")
else:
    print("[LOAD_BASELINE] Using pre-computed initial metrics from disk.")

[LOAD_BASELINE] Using pre-computed initial metrics from disk.


## Analyze

In [6]:
importlib.reload(analysis_module)
for result in sector_results:
    analysis = analyze(result['generated'], service_files=result.get('service_files'), benchmark_dir=BENCHMARK_DIR)
    result['analysis'] = analysis
    print(f"Analysis - [{result['sector_name']}] Query {result['query_index']}\n", analysis)

Analysis - [01-energy] Query 1
 AST: No syntax errors found.
Ruff: F841 Local variable `equipment_data` is assigned to but never used
Ruff:  --> /var/folders/57/p3z5bqsd11l937yr1409c3m00000gn/T/tmpf66j3pqf/generated.py:6:5
Ruff:   |
Ruff: 4 |     # Retrieve equipment status and performance metrics
Ruff: 5 |     equipment_response = requests.get('https://api.example.com/equipment-status')
Ruff: 6 |     equipment_data = equipment_response.json()
Ruff:   |     ^^^^^^^^^^^^^^
Ruff: 7 |
Ruff: 8 |     # Access active alerts for system performance issues
Ruff:   |
Ruff: help: Remove assignment to unused variable `equipment_data`
Ruff: F841 Local variable `alerts_data` is assigned to but never used
Ruff:   --> /var/folders/57/p3z5bqsd11l937yr1409c3m00000gn/T/tmpf66j3pqf/generated.py:10:5
Ruff:    |
Ruff:  8 |     # Access active alerts for system performance issues
Ruff:  9 |     alerts_response = requests.get('https://api.example.com/alerts')
Ruff: 10 |     alerts_data = alerts_response.json(

## Refinement LLM Call

In [7]:
def _call_refined(result):
    original_code = result['generated'].removesuffix('\n\ncompose()')
    refined_prompt = f'''{result['prompt']}

You are given a review of the code you generated and the code itself.
Use the review to improve the code.

## Analysis
{result['analysis']}

## Original Code
{original_code}

'''
    generated_refined = call_llm(refined_prompt, MODEL, 'Return only Python code, no explanation.')
    generated_refined += '\n\ncompose()'
    result['generated_refined'] = generated_refined
    return result

refined_count = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_call_refined, r): r for r in sector_results}
    for future in as_completed(futures):
        future.result()
        refined_count += 1
        print(f"[{refined_count}/{len(sector_results)}] Refinement done")

[1/50] Refinement done
[2/50] Refinement done
[3/50] Refinement done
[4/50] Refinement done
[5/50] Refinement done
[6/50] Refinement done
[7/50] Refinement done
[8/50] Refinement done
[9/50] Refinement done
[10/50] Refinement done
[11/50] Refinement done
[12/50] Refinement done
[13/50] Refinement done
[14/50] Refinement done
[15/50] Refinement done
[16/50] Refinement done
[17/50] Refinement done
[18/50] Refinement done
[19/50] Refinement done
[20/50] Refinement done
[21/50] Refinement done
[22/50] Refinement done
[23/50] Refinement done
[24/50] Refinement done
[25/50] Refinement done
[26/50] Refinement done
[27/50] Refinement done
[28/50] Refinement done
[29/50] Refinement done
[30/50] Refinement done
[31/50] Refinement done
[32/50] Refinement done
[33/50] Refinement done
[34/50] Refinement done
[35/50] Refinement done
[36/50] Refinement done
[37/50] Refinement done
[38/50] Refinement done
[39/50] Refinement done
[40/50] Refinement done
[41/50] Refinement done
[42/50] Refinement done
[

## Evaluate Refined

In [8]:
for result in sector_results:
    refined_metrics = evaluate(result['generated_refined'], result['query'].get('endpoints', []))
    result['refined_metrics'] = refined_metrics
    print_metrics(refined_metrics, f"Refined evaluation - [{result['sector_name']}] Query {result['query_index']}")

Refined evaluation - [01-energy] Query 1
  Precision: 1.00
  Recall:    1.00
  F1:        1.00
  Extracted: ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Expected:  ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Missing:   []
  Extra:     []
Refined evaluation - [01-energy] Query 2
  Precision: 1.00
  Recall:    1.00
  F1:        1.00
  Extracted: ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Expected:  ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Missing:   []
  Extra:     []
Refined evaluation - [01-energy] Query 3
  Precision: 0.75
  Recall:    0.75
  F1:        0.75
  Extracted: ['GET /electricity-demand', 'GET /energy-patterns', 'GET

## Save Outputs

In [9]:
import os
from datetime import datetime
importlib.reload(output_module)

if LOAD_BASELINE:
    baseline_tag = os.path.basename(BASELINE_DIR)
    mode_tag = f"refined_from_{baseline_tag}"
else:
    mode_tag = "refined"

run_name = datetime.now().strftime('%Y-%m-%d_%H-%M-%S') + f"_{mode_tag}"
outdir = output_module.make_output_dir(run_name)
for r in sector_results:
    output_module.write_query_output(outdir, r)

run_config = {
    'benchmark_dir': str(BENCHMARK_DIR),
    'benchmark_types': BENCHMARK_TYPES,
    'benchmark_limit': BENCHMARK_LIMIT,
    'query_limit': QUERY_LIMIT,
    'baseline': False,
    'model': MODEL,
    'load_baseline': LOAD_BASELINE,
    'baseline_dir': BASELINE_DIR if LOAD_BASELINE else None,
}
output_module.write_overall_summary(outdir, sector_results, run_config)
print('Wrote outputs to', outdir)

Wrote outputs to output/2026-06-16_16-03-08_refined_from_2026-06-16_15-57-18_baseline


## Summary

In [10]:
print_evaluation_summary([result['initial_metrics'] for result in sector_results], "Initial Evaluation Summary")
print_evaluation_summary([result['refined_metrics'] for result in sector_results], "Refined Evaluation Summary")

Initial Evaluation Summary
  Average Precision: 0.53
  Average Recall:    0.63
  Average F1:        0.56
  Avg. Missing Endpoints: 1.60
  Avg. Extra Endpoints:   2.78
  Correct Compositions: 5/50 (10.0%)
Refined Evaluation Summary
  Average Precision: 0.54
  Average Recall:    0.66
  Average F1:        0.58
  Avg. Missing Endpoints: 1.54
  Avg. Extra Endpoints:   2.82
  Correct Compositions: 5/50 (10.0%)
